# Machine Learning Model Comparison

This notebook compares three baseline classification models for predicting whether an ETF's 5-trading-day future return is positive. It uses the leakage-safe ML dataset and purged chronological split. It does not write predictions to the database and does not save production model files.

## 1. Setup

Load shared dataset and training utilities from the backend. The database connection uses existing environment-based configuration.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve
from sqlalchemy import create_engine

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKEND_ROOT = PROJECT_ROOT / "backend"
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.core.config import settings
from app.ml.features import (
    FEATURE_COLUMNS,
    build_ml_dataset,
    load_ml_source_data,
    purged_chronological_split,
    validate_purged_split,
)
from app.ml.training import (
    compare_models,
    evaluate_model,
    evaluate_models_per_etf,
    feature_importance_mapping,
    majority_baseline_accuracy,
    predict_positive_probability,
    train_models,
)

plt.style.use("default")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

engine = create_engine(settings.database_url, pool_pre_ping=True)
FEATURE_COLUMNS

## 2. Load Leakage-Safe ML Dataset

The target is `1` when the ETF's 5-trading-day future return is positive, and `0` otherwise. Features use only information available at time `t`; future return and target are not used as features.

In [ ]:
source_df = load_ml_source_data(engine)
ml_df = build_ml_dataset(source_df, horizon_days=5)
print("source shape:", source_df.shape)
print("ML dataset shape:", ml_df.shape)
ml_df.head()

## 3. Apply Purged Chronological Split

A random split is not appropriate for financial time-series data because it can train on observations that occur after test observations. Since the label uses a 5-trading-day future return, rows immediately before the test period are purged if their target horizon reaches into the test period. This keeps the test set untouched.

In [ ]:
unique_dates = pd.Series(sorted(ml_df["date"].unique()))
cutoff_index = int(len(unique_dates) * 0.8)
test_start_date = unique_dates.iloc[cutoff_index]

train_df, test_df, purged_df = purged_chronological_split(
    ml_df,
    test_start_date=test_start_date,
    horizon=5,
)
validate_purged_split(train_df, test_df, test_start_date=test_start_date)

split_summary = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "min_date": train_df["date"].min(), "max_date": train_df["date"].max()},
        {"split": "purged", "rows": len(purged_df), "min_date": purged_df["date"].min(), "max_date": purged_df["date"].max()},
        {"split": "test", "rows": len(test_df), "min_date": test_df["date"].min(), "max_date": test_df["date"].max()},
    ]
)

print("test_start_date:", test_start_date)
split_summary

## 4. Train/Test Distributions

Check class distributions before modeling. Direction prediction is difficult in financial data, so class balance and baseline performance must be visible.

In [ ]:
train_distribution = train_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="train_rows")
test_distribution = test_df["target"].value_counts().sort_index().rename_axis("target").reset_index(name="test_rows")

display(train_distribution)
display(test_distribution)
display(train_df.groupby("symbol").size().reset_index(name="train_rows"))
display(test_df.groupby("symbol").size().reset_index(name="test_rows"))

## 5. Majority-Class Baseline

A baseline is necessary because a model can look acceptable by accuracy alone when one class is more common. The majority-class baseline predicts the most frequent class in the test set for every test row. Models should be compared against this simple benchmark and evaluated using balanced accuracy, F1, and ROC-AUC as well.

In [ ]:
baseline = majority_baseline_accuracy(train_df, test_df)
print(f"Majority class in test set: {baseline.majority_class}")
print(f"Majority-class baseline accuracy: {baseline.majority_baseline_accuracy:.4f}")

## 6. Train Logistic Regression, Random Forest, and XGBoost

The Logistic Regression model uses a `StandardScaler` fitted only on the training data. Tree-based models are trained without scaling. No model is saved here.

In [ ]:
models = train_models(train_df, random_state=42)
list(models.keys())

## 7. Comparison Metrics Table

Evaluate on the untouched test set. The table is sorted by ROC-AUC for readability, but the final model choice should also consider balanced accuracy, F1-score, ETF-level stability, explainability, and simplicity.

In [ ]:
comparison = compare_models(models, test_df)
comparison

## 8. Confusion Matrices

Confusion matrices show the types of classification mistakes each model makes.

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(14, 4))
for ax, (model_name, model) in zip(axes, models.items()):
    matrix = np.array(evaluate_model(model_name, model, test_df).confusion_matrix)
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(model_name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    for row in range(2):
        for col in range(2):
            ax.text(col, row, matrix[row, col], ha="center", va="center")
fig.colorbar(image, ax=axes, fraction=0.02, pad=0.04)
plt.tight_layout()
plt.show()

## 9. ROC Curves

ROC curves compare each model's ability to rank positive examples above negative examples across thresholds. Thresholds are not tuned in this notebook.

In [ ]:
fig, ax = plt.subplots()
for model_name, model in models.items():
    probabilities = predict_positive_probability(model, test_df)
    fpr, tpr, _ = roc_curve(test_df["target"], probabilities)
    roc_auc = comparison.loc[comparison["model"] == model_name, "roc_auc"].iloc[0]
    ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random")
ax.set_title("ROC Curves")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Per-ETF Performance

Evaluate the same global models separately by ETF. This helps identify whether performance is consistent or concentrated in only one ETF.

In [ ]:
per_etf_metrics = evaluate_models_per_etf(models, test_df)
per_etf_metrics

## 11. Feature Importance and Coefficients

For Random Forest and XGBoost, report built-in feature importances. For Logistic Regression, report coefficients after scaling. SHAP is intentionally not used at this stage.

In [ ]:
for model_name, model in models.items():
    print(model_name)
    display(feature_importance_mapping(model_name, model))

## 12. Discussion

Financial direction prediction is difficult because prices are noisy, regimes change, and technical indicators describe recent behavior rather than guaranteeing future returns. A preferred model should not be chosen by accuracy alone. Consider ROC-AUC, balanced accuracy, F1-score, consistency across ETFs, explainability, and implementation simplicity. If all models perform weakly relative to the majority-class baseline, that result should be reported honestly rather than hidden.